# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiment: **E007-dinov2-stack** — the deliberate multi-lever run: DINOv2
ViT-S/14 (at 224 via position-embedding interpolation) fine-tuned end-to-end on
2.5D adjacent-slice triplets from a fixed 140mm crop, per-label attention heads,
tier-weighted loss. Attribution is knowingly traded for speed: each lever is
individually evidenced (forum ablations, E004/E005 readouts, Ryan's methodology),
and the internal control is the staged unfreeze itself — the frozen warm-up
epochs' val score IS the frozen-DINO-triplet baseline on the same split. Heads
cold-start (E005's resnet heads don't fit 384-d ViT features). One decode pass
(~75 min), then three per-plane fine-tunes. Requires the `WANDB_API_KEY` secret,
the `knee-labels` dataset, and internet (DINOv2 weights download at train time).

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
# STALE for E007 — bump to the e007 squash merge before `kaggle kernels push`
# (this run needs the ViT image_size override, absent at cdbf23d).
COMMIT = "cdbf23d"
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_blended import BLENDED_LABEL_SOURCE

In [ ]:
# Competition data (DICOMs + series metadata) is pre-mounted; blended soft labels
# (with the per-cell __weight companions for E006a) via the knee-labels dataset;
# the E005 winner's feature bank + checkpoints via knee-e005-artifacts.
from pathlib import Path

from knee.data import load_blended_labels, weight_matrix

SLUG = "rsna-knee-abnormality-detection"
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)


def find_input(name: str, filename: str) -> Path:
    bases = [Path("/kaggle/input") / name, Path("/kaggle/input/datasets/josiemachalek") / name]
    for base in bases:
        if (base / filename).exists():
            return base / filename
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{name}/{filename} not found; mounts: {listing}")


labels = load_blended_labels(find_input("knee-labels", "blended_labels_v1.csv"), include_weights=True)
weights = weight_matrix(labels)
print(f"blended labels: {len(labels)} studies; weights {weights.shape}")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(
        project="rsna-knee",
        config={"commit": COMMIT, "label_source": BLENDED_LABEL_SOURCE, "n_label_studies": len(labels)},
    )
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E007 config — best-evidence choices, set directly rather than gated on E005:
# crop140 (forum-measured geometry win), attention heads (targets the thin-structure
# floor), tier weights (Ryan's methodology; team call), 224px ViT (fine-tuning at
# 518 needs a ~100GB pixel cache; E004's null was about FROZEN features at 518 —
# the 0.87+ pipelines fine-tune at 224-392).
from knee.model import DINOV2_BACKBONE, HeadType

SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
BACKBONE = DINOV2_BACKBONE
INPUT_SIZE = 224  # cache/slice resolution AND the ViT's overridden image_size
CROP_MM = 140.0
HEAD_TYPE = HeadType.ATTENTION
USE_WEIGHTS = True

CHECKPOINT_DIR = Path("/kaggle/working")
CACHE_DIR = Path("/tmp/pixel_cache")  # ephemeral: only checkpoints persist as output

In [ ]:
# Fixed 90/10 split — the E006/E007 regime marker. Same seed always, so every
# fine-tune era experiment shares the split and stays comparable.
import numpy as np

from knee.cv import stratified_holdout
from knee.labels import LABEL_COLUMNS

label_matrix = labels[list(LABEL_COLUMNS)].to_numpy(dtype=np.float32)
val_mask = stratified_holdout(label_matrix, val_fraction=0.1, seed=0)
print(f"split: {int((~val_mask).sum())} train / {int(val_mask.sum())} val")

In [ ]:
# E007: pixel cache at the stack geometry (~75 min decode), then three per-plane
# fine-tunes (~30-40 min each). Heads cold-start; the 2 frozen warm-up epochs'
# printed val scores are the frozen-DINO-triplet baseline on this same split.
# After the loop, the fine-tuned ENSEMBLE scores on the split — the headline number.
from knee.finetune import (
    FinetuneConfig,
    build_pixel_cache,
    evaluate_ensemble_holdout,
    finetune_plane,
)
from knee.model import InputMode, KneeModel, load_model

cache = build_pixel_cache(
    COMP_ROOT, labels, CACHE_DIR, series_types=SERIES_TYPES,
    input_size=INPUT_SIZE, crop_mm=CROP_MM,
)
print("cache coverage:", {t.value: n for t, n in cache.coverage.items()})

finetune_results, finetuned_models = {}, {}
for series_type in SERIES_TYPES:
    model = KneeModel(
        BACKBONE, head_type=HEAD_TYPE, input_mode=InputMode.TRIPLETS, image_size=INPUT_SIZE
    )
    result = finetune_plane(
        CACHE_DIR, label_matrix, val_mask,
        series_type=series_type, model=model,
        out_path=CHECKPOINT_DIR / f"e007_{series_type.value}.pt",
        config=FinetuneConfig(),
        input_size=INPUT_SIZE, crop_mm=CROP_MM,
        cell_weights=weights if USE_WEIGHTS else None,
        label_source=BLENDED_LABEL_SOURCE,
    )
    finetune_results[series_type] = result
    finetuned_models[series_type] = load_model(result.checkpoint_path).model
    print(f"{series_type.value}: best per-plane val macro {result.best_val_macro_auc:.3f} (epoch {result.best_epoch + 1})")

ensemble_val = evaluate_ensemble_holdout(CACHE_DIR, finetuned_models, label_matrix, val_mask)
ensemble_macro = float(np.nanmean(list(ensemble_val.values())))
print(f"E007 FINETUNED ensemble val macro: {ensemble_macro:.3f}")
print({label: round(auc, 3) for label, auc in ensemble_val.items()})

In [ ]:
# The three e007_*.pt checkpoints persist as notebook output. Submission gate
# (team policy after E004): publish to knee-weights + run inference only if the
# holdout ensemble macro clears the best CV ever recorded (0.783) decisively —
# otherwise record the numbers in experiments.md and bank the learning. Note the
# holdout and blended-cv are different protocols; compare direction and magnitude,
# not decimals.
import math

if run is not None:
    run.config.update({
        "backbone": BACKBONE, "input_size": INPUT_SIZE, "crop_mm": CROP_MM,
        "head_type": HEAD_TYPE.value, "tier_weighted": USE_WEIGHTS,
        "input_mode": "triplets",
    })
    wandb.log({
        "holdout/finetuned_macro": ensemble_macro,
        **{f"holdout/finetuned/{label}": auc for label, auc in ensemble_val.items() if not math.isnan(auc)},
        **{
            f"finetune/{series_type.value}/best_val_macro": result.best_val_macro_auc
            for series_type, result in finetune_results.items()
        },
    })
    run.finish()